# Reading and Writing Files

Pyspark provides powerful and flexible API to **Read** and **Write** data from a variety of sources including - **CSV**, **JSON**, **Parquet**, **ORC** and **Databases** - using Spark Dataframes interface . These operations from the backbone of most ETL pipelines. enabling the process data at scale in distributed environments.

In this tutorial , we will learn the general pattern for reading and writing files in pyspark. 

Lets get started with reading files. The general format for reading files in pyspark  is all follows:


```
from pyspark.sql import SparkSession

# Step 1: Initialize SparkSession
spark = SparkSession.builder \
    .appName("Read File Example") \
    .getOrCreate()

# Step 2: Read data
df = spark.read \
    .format("<file_format>") \
    .option("<option_name>", "<option_value>") \
    .load("<path_to_file_or_folder>")

# Step 3: Inspect data
df.show(5)
df.printSchema()


```

## Read Modes Explained

### Parameters Explained

|Parameter| Description |Example|
|---------|-------------|-------|
|<file_format>|Type of data source|csv,json,parquet, orc, jdbc|
|option|Used to configure reading options|.options("header","true")or .option("inferschema","true")|
|load()|path fo file or directory| /datasets/customer.csv|


####  Example 1 : Reading CSV File

In [0]:

df_csv = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/datasets/customers.csv")


#### Example 2: Reading a Parquet file


In [0]:
df_parquet = spark.read \
    .format("parquet") \
    .load("/datasets/sales.parquet")


#### Example 3: Reading a JSON File

In [0]:

df_json = spark.read \
    .format("json") \
    .load("/datasets/events.json")


#### Example 4: Reading from a Hive Table
You can also read data from Hive tables using spark.read.table() or SQL queries.

In [0]:
# Reading a Hive table customers in default schema into a DataFrame
df_hive = spark.read.table("default.customers")

df_hive.show(5)
df_hive.printSchema()


In [0]:
# Querying a Hive table with SQL
df_hive_sql = spark.sql("SELECT id, name, email FROM default.customers WHERE country='US'")
df_hive_sql.show(5)

#### Example 5 : Reading from Database(JDBC)





In [0]:
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/mydb") \
    .option("dbtable", "public.customers") \
    .option("user", "postgres") \
    .option("password", "mypassword") \
    .load()


### Shortcuts for common Formats

For CSV, JSON, and Parquet, you can skip `.format()` and directly use built-in methods. You will see this sytax being used commonly.

In [0]:
df_csv = spark.read.csv(path, header=True, inferSchema=True)
df_json = spark.read.json(path)
df_parquet = spark.read.parquet(path)


### Read Modes Explained
Read modes handle malformed or corrupt records during data loading from formats like JSON, CSV, or Parquet. These modes determine whether to process bad data leniently, skip it, or fail the job.


|Mode|Description|
|----|-----------|
|PERMISSIVE|	Default mode; sets corrupt record fields to null and stores the raw data in _corrupt_record column.|
|DROPMALFORMED|Drops entire rows with any schema mismatch or corruption, loading only valid records.|
|FAILFAST|Throws an exception immediately upon encountering the first malformed record.|

Set these via `.option("mode", "PERMISSIVE")` or `.mode("DROPMALFORMED")` on the DataFrameReader.